# C3 · Civic Trust and Knowledge Verification Engine: PP1 demonstration

Run the cells **top to bottom**. Each cell is one part of the demonstration.
The same demo is available in a terminal: `python -m app.cli.demo --pause`.

**What is real here:** every algorithm, the database, the pretrained embedding model, the translation step.
**What is mocked:** C1 (the free-text messages come from a file), C4 (the contract `feedback` messages come from a file)
and C2 (the knowledge graph is a JSON file in the shape of C2's `fact` contract). Integration is planned; the interfaces follow the agreed contract.

In [ ]:
import os, sys, types
from pathlib import Path

# run from components/c3-trust/service so .env and the app package are found
HERE = Path.cwd()
SERVICE = next(p for p in [HERE, HERE / "service", HERE.parent.parent / "service", HERE.parent / "service"] if (p / "app").is_dir())
os.chdir(SERVICE); sys.path.insert(0, str(SERVICE))

USE_SQLITE = os.environ.get("C3_DEMO_SQLITE") == "1"        # True = no Docker needed
EMBEDDING = os.environ.get("C3_DEMO_EMBEDDING") or None       # None = the real model; "hash" = plumbing check only

from app.cli import demo
from app.cli._common import build, utf8
from app.core.evaluate import load_dataset
utf8()
args = types.SimpleNamespace(sqlite=USE_SQLITE, db=None, embedding=EMBEDDING, pause=False, detail=None, path="freetext", part="all")
pipe = build(args, reset=True)            # recreates C3's own tables only
pipe.load_reporters(); pipe.sync_facts()
ds = load_dataset(pipe.s.data_dir)
print("ready:", pipe.embedder.name)

## 1 · Set-up: C3 owns the confidence of every fact
C2's graph stores the fact. C3 stores how far to believe it.

In [ ]:
demo.part_setup(pipe, args)

## 2 · Fact grounding: which stored fact is the citizen disputing?
English, Sinhala and Singlish messages are turned into English first, then compared with the stored facts.

In [ ]:
demo.part_grounding(pipe, ds, args)

## 3 · Confidence story: the passport fee
Reports arrive through both paths. The percentage is always shown; the warning waits for repeated drops.

In [ ]:
demo.part_story(pipe, ds, args)

## 4 · Citizens who disagree with each other
The system records the disagreement and does not pick a winner.

In [ ]:
demo.part_conflict(pipe, ds, args)

## 5 · The official source changes a value
A new fact version starts with fresh confidence; reporters are judged against the official value.

In [ ]:
demo.part_version(pipe, ds, args)

## 6 · What C4 receives
The contract `trust` message (valid against the team's schema) and C3's private explanation.

In [ ]:
demo.part_exposure(pipe, ds, args)

## 7 · Do the formulas behave as claimed?
Live property checks, including the calibration-by-design check: three perfect reports land exactly on the flag level.

In [ ]:
demo.part_checks(pipe, ds, args)

## Chart
The passport story chart is saved by part 3 to `research/experiments/out/passport_fee_story.svg`.

In [ ]:
from IPython.display import SVG, display
chart = pipe.s.out_dir / "passport_fee_story.svg"
display(SVG(filename=str(chart))) if chart.exists() else print("run part 3 first")